# 05 — Solution Representation and Fitness Evaluation

**Maps to:** `report/Chapters/Task2.tex` §`T2:Representation`.  
**Ticket:** TICKET-05.

- permutation encoding;
- `tour_length(chromosome, dist_matrix)` for fitness;
- `is_valid_permutation` for invariants.

## Permutation Encoding

In this project, a TSP solution (chromosome) is represented as a **permutation** of city indices. Each chromosome is a one-dimensional array of length `n`, where `n` is the number of cities. Each city index appears exactly once, and the tour is interpreted as a closed loop, where the salesman visits cities in the order given and returns to the starting city.

In [4]:
import numpy as np
import sys
sys.path.insert(0, '..')

from pathlib import Path
import tsplib95

In [5]:
def tour_length(chromosome, dist_matrix):
    """
    Return total distance of a closed tour.
    chromosome:  1-D array of city indices, length n.
    dist_matrix: 2-D array of shape (n, n).
    """
    n = len(chromosome)
    total = 0.0
    for i in range(n):
        city_a = chromosome[i]
        city_b = chromosome[(i + 1) % n]
        total += dist_matrix[city_a, city_b]
    return total


def is_valid_permutation(chromosome, n_cities):
    """Return True if chromosome is a valid permutation of 0..n_cities-1."""
    return (
        len(chromosome) == n_cities
        and len(set(chromosome)) == n_cities
        and set(chromosome) == set(range(n_cities))
    )

Two functions are implemented as shown at the above:

- `tour_length(chromosome, dist_matrix)` — computes the total distance of a closed tour by summing the distances between consecutive cities, including the return edge from the last city back to the first.

- `is_valid_permutation(chromosome, n_cities)` — checks whether a chromosome is a valid TSP tour, meaning it visits every city exactly once with no duplicates or missing cities.

## Unit Test

The functions are verified using a hand-built 4-city square where the correct answers are known in advance. The four cities are placed at coordinates (0,0), (1,0), (1,1), and (0,1), forming a unit square with side length 1.0. The tour 0 → 1 → 2 → 3 → 0 traces the full perimeter, giving a known total distance of 4.0.

If `tour_length` returns any value other than 4.0, the assertion will raise an error immediately, catching any implementation bugs before they propagate into the GA.

In [6]:
# build 4-city square manually
square_coords = np.array(
    [[0, 0], [1, 0], [1, 1], [0, 1]],
    dtype=np.float64
)
diff        = square_coords[:, np.newaxis, :] - square_coords[np.newaxis, :, :]
square_dist = np.sqrt((diff ** 2).sum(axis=-1))

# tour: 0 → 1 → 2 → 3 → 0, perimeter = 4.0
tour   = np.array([0, 1, 2, 3])
result = tour_length(tour, square_dist)

try:
    assert abs(result - 4.0) < 1e-9, f"Expected 4.0, got {result}"
    assert is_valid_permutation(tour, 4)
    assert not is_valid_permutation(np.array([0, 1, 1, 3]), 4)
    print("Unit test passed.")
except AssertionError as e:
    print(f"Unit test does not passed: {e}")

Unit test passed.


## Demo on kroA100


The implemented functions are applied to kroA100 to verify that they work correctly on a real TSP instance. A random tour is generated using a fixed seed of 42, ensuring the result is reproducible across runs. This random tour serves as a naive baseline, where it visits all 100 cities in a random order without any optimisation, representing the worst-case starting point before the Genetic Algorithm begins.

In [7]:
def load_tsp(path):
    problem    = tsplib95.load(str(path))
    nodes      = list(problem.get_nodes())
    coords     = np.array(
        [problem.node_coords[node] for node in nodes],
        dtype=np.float64
    )
    diff        = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))
    return coords, dist_matrix

coords, dist_matrix = load_tsp(Path('../data/TSP-dataset/kroA100.tsp'))

# random tour as a demo baseline
rng         = np.random.default_rng(42)
random_tour = rng.permutation(len(coords))

print(f"Tour length          : {tour_length(random_tour, dist_matrix):.2f}")
print(f"Is valid permutation : {is_valid_permutation(random_tour, len(coords))}")

Tour length          : 166578.15
Is valid permutation : True
